# Commercial Banking Email Classification POC Testing

Use this notebook to test the Ollama Qwen + OKF classification pipeline before adding UI or review persistence.

In [1]:
from pathlib import Path

from app.core.settings import get_settings
from app.core.dspy_config import configure_dspy
from app.classification.okf_loader import load_category_documents, load_issue_catalog
from app.classification.pipeline import run_email_classification
from app.classification.evaluation import evaluate_samples

settings = get_settings()
configure_dspy(settings)
settings.dspy_model_identifier

'ollama_chat/qwen2.5:3b'

In [2]:
catalog = load_issue_catalog()
categories = load_category_documents()
len(categories), catalog[:300]

(20,
 '---\ntype: catalog\ntitle: Commercial Banking Operations Category Catalog\nversion: 1.0\nstatus: active\n---\n\n# Commercial Banking Operations Category Catalog\n\n## Customer Maintenance\n\n- CAT-001 Registered Email Change\n- CAT-002 Registered Phone Number Change\n- CAT-005 Company Address Update\n- CAT-006 Co')

In [3]:
example = {
    "subject": "Change of registered mobile number",
    "body": "Hello Team, our finance manager has changed his mobile number. Please update the registered mobile number for our current account with immediate effect. The old number should no longer be used for OTP alerts or banking communication.",
}

result = run_email_classification(**example)
result.model_dump()

{'final_category_id': 'CAT-002',
 'final_category_name': 'Registered Phone Number Change',
 'confidence': 0.95,
 'needs_review': False,
 'routing_summary': {'primary_intent': "Update the registered mobile number for the customer's current account.",
  'business_domain': 'Commercial Banking Operations',
  'requested_action': "The bank operations team should update the customer's registered mobile number in their system.",
  'candidate_categories': ['CAT-002 Registered Phone Number Change',
   'CAT-019 Account Statement Request (though this is not explicitly mentioned in the email)'],
  'evidence_phrases': ['"Please update the registered mobile number for our current account with immediate effect."'],
  'body_sufficient_for_classification': 'true',
  'routing_confidence': 0.85},
 'candidates': [{'category_id': 'CAT-002',
   'category_name': 'Registered Phone Number Change',
   'score': 51.0,
   'source_file': 'knowledge/commercial_banking/categories/cat_002_registered_phone_number_change

In [4]:
medium_example = {
    "subject": "Payment not received by beneficiary",
    "body": "Dear Team, we made a NEFT payment to our supplier yesterday morning from our current account. The amount has been debited from our account, but the supplier says they have not received the funds yet. Please check the status of this payment and confirm whether it has been processed or returned.",
}

medium_result = run_email_classification(**medium_example)
medium_result.model_dump()

{'final_category_id': 'CAT-016',
 'final_category_name': 'Payment Status Inquiry',
 'confidence': 0.95,
 'needs_review': False,
 'routing_summary': {'primary_intent': 'Payment Status Inquiry',
  'business_domain': 'Commercial Banking Operations',
  'requested_action': 'Investigate and confirm the status of the NEFT payment to ensure it has been processed correctly and inform the customer if there are any issues with the transaction.',
  'candidate_categories': ['CAT-016 Payment Status Inquiry',
   'CAT-017 Failed Transaction Investigation'],
  'evidence_phrases': ['"we made a NEFT payment to our supplier yesterday morning from our current account. The amount has been debited from our account',
   'but the supplier says they have not received the funds yet."'],
  'body_sufficient_for_classification': 'true',
  'routing_confidence': 0.95},
 'candidates': [{'category_id': 'CAT-016',
   'category_name': 'Payment Status Inquiry',
   'score': 44.0,
   'source_file': 'knowledge/commercial_ban

In [5]:
hard_example = {
    "subject": "Need help with account updates and payment issue",
    "body": "Hello, our office contact details have changed and we also need help because one recent vendor payment did not go through as expected. Please update your records and advise what needs to be done for the transaction. We can share more details if required.",
}

hard_result = run_email_classification(**hard_example)
hard_result.model_dump()

{'final_category_id': 'CAT-002',
 'final_category_name': 'Registered Phone Number Change',
 'confidence': 0.85,
 'needs_review': True,
 'routing_summary': {'primary_intent': 'The customer wants assistance with updating their account details and resolving an unpaid vendor payment.',
  'business_domain': 'Commercial Banking Operations',
  'requested_action': 'To provide guidance on how to update the account information and investigate the failed payment transaction.',
  'candidate_categories': ['CAT-016 Payment Status Inquiry',
   'CAT-017 Failed Transaction Investigation',
   'CAT-001 Registered Email Change',
   'CAT-002 Registered Phone Number Change'],
  'evidence_phrases': ['"Hello',
   'our office contact details have changed and we also need help because one recent vendor payment did not go through as expected. Please update your records and advise what needs to be done for the transaction."'],
  'body_sufficient_for_classification': 'true',
  'routing_confidence': 0.85},
 'candid

In [8]:
sample_path = Path("./data/email_classification_samples.jsonl")
evaluation = evaluate_samples(sample_path)
evaluation

{'total': 3,
 'final_accuracy': 0.6666666666666666,
 'candidate_recall': 1.0,
 'results': [{'email_id': 'EMAIL-001',
   'expected_category_id': 'CAT-002',
   'final_category_id': 'CAT-002',
   'final_match': True,
   'candidate_hit': True,
   'needs_review': False},
  {'email_id': 'EMAIL-002',
   'expected_category_id': 'CAT-016',
   'final_category_id': 'CAT-016',
   'final_match': True,
   'candidate_hit': True,
   'needs_review': False},
  {'email_id': 'EMAIL-003',
   'expected_category_id': 'CAT-020',
   'final_category_id': 'CAT-001',
   'final_match': False,
   'candidate_hit': True,
   'needs_review': False}]}